# Qwen3 Abstract Evaluator (Modular)

This notebook orchestrates reusable utilities from:
- `experiments/utils` (model-agnostic)
- `experiments/qwen/utils` (Qwen3-specific)

It supports:
- train/val/test load + clean/validate
- Qwen3 chat message construction
- JSONL export + HF datasets
- configurable eval every `epoch` or `steps`
- early stopping
- per-epoch JSON metrics artifacts
- post-training adapter eval + base-model eval
- W&B integration


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd


def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "experiments").exists() and (p / "data").exists():
            return p
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


In [ ]:
from experiments.qwen.utils.chat import add_messages_and_targets
from experiments.qwen.utils.configs import build_data_paths, build_qwen3_default_config
from experiments.qwen.utils.modeling import load_qwen3_model_for_inference
from experiments.qwen.utils.pipeline import (
    estimate_qwen_token_percentiles,
    evaluate_base_model,
    evaluate_saved_adapters,
    evaluate_single_adapter,
)
from experiments.qwen.utils.training import train_qwen3

from experiments.utils.data import (
    clean_train_val_test,
    load_train_val_test_dfs,
    score_distribution,
)
from experiments.utils.datasets_io import export_split_jsonl, to_hf_dataset_dict
from experiments.utils.evaluation import parse_rationale, parse_score
from experiments.utils.generation import generate_predictions_from_messages
from experiments.utils.logging_utils import setup_logger
from experiments.utils.runtime import configure_wandb_dir, set_global_seed



In [ ]:
cfg = build_qwen3_default_config(PROJECT_ROOT)

# --- Core run identity ---
cfg.model_name = "Qwen/Qwen3-8B"
cfg.run_name = "qwen3_8b_abstract_evaluator_lora_speed_modular"

# --- Data paths (edit if you want different splits) ---
cfg.data_paths = build_data_paths(
    train_path=PROJECT_ROOT / "data/data/train/all.jsonl",
    val_path=PROJECT_ROOT / "data/data/val/all.jsonl",
    test_path=PROJECT_ROOT / "data/data/test/all.jsonl",
)

# --- Trainer config (speed-first + train-until-no-improvement pattern) ---
cfg.max_seq_length = 2048
cfg.use_4bit = False                         # full LoRA (no QLoRA), better stability/quality
cfg.train.num_train_epochs = 50                  # high ceiling; early stopping decides actual stop
cfg.train.per_device_train_batch_size = 8        # safer: keeps effective batch near prior setup
cfg.train.per_device_eval_batch_size = 24
cfg.train.gradient_accumulation_steps = 1          # 8 * 1 ~= prior effective batch (4 * 2)
cfg.train.learning_rate = 8e-5
cfg.train.logging_steps = 5

# Eval/save once per epoch (lighter overhead)
cfg.train.eval_strategy = "epoch"
cfg.train.eval_steps = None
cfg.train.save_strategy = "epoch"
cfg.train.save_steps = None
cfg.train.save_total_limit = 20              # keep enough resume checkpoints while training

# Stop when validation loss no longer improves
cfg.train.early_stopping_patience = 3
cfg.train.early_stopping_threshold = 0.0

# Throughput-focused settings
cfg.train.dataloader_num_workers = 8
cfg.train.dataloader_pin_memory = True
cfg.train.auto_find_batch_size = True
cfg.train.tf32 = True
cfg.train.gradient_checkpointing = False

# --- Generation/eval config ---
cfg.generation.max_new_tokens = 120
cfg.generation.batch_size = cfg.train.per_device_eval_batch_size

# --- W&B ---
cfg.wandb.enabled = True
cfg.wandb.project = "abstract-evaluator-qwen3-sft"
cfg.wandb.entity = None
cfg.wandb.tags = ["qwen3", "lora", "sft", "modular", "full-lora"]

cfg.ensure_dirs()
cfg.as_dict()









In [ ]:
set_global_seed(cfg.seed)
configure_wandb_dir(str(cfg.wandb.dir))

log_dir = cfg.output_root / "logs"
logger = setup_logger(
    name=f"{cfg.run_name}_pipeline",
    log_dir=log_dir,
    log_file=f"{cfg.run_name}.log",
)
logger.info("Initialized run config: %s", json.dumps(cfg.as_dict(), ensure_ascii=False))
log_dir


In [ ]:
train_df, val_df, test_df = load_train_val_test_dfs(
    train_path=cfg.data_paths.train_path,
    val_path=cfg.data_paths.val_path,
    test_path=cfg.data_paths.test_path,
)

train_df, val_df, test_df = clean_train_val_test(train_df, val_df, test_df)

print("Shapes:", train_df.shape, val_df.shape, test_df.shape)
print("Train score dist:", score_distribution(train_df))
print("Val score dist:", score_distribution(val_df))
print("Test score dist:", score_distribution(test_df))


In [ ]:
train_df = add_messages_and_targets(train_df)
val_df = add_messages_and_targets(val_df)
test_df = add_messages_and_targets(test_df)

print(json.dumps(train_df.iloc[0]["messages"], indent=2, ensure_ascii=False)[:2000])


In [ ]:
jsonl_paths = export_split_jsonl(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    output_dir=cfg.output_root / "jsonl",
)

ds = to_hf_dataset_dict(train_df, val_df, test_df)

print("JSONL paths:", jsonl_paths)
print(ds)


In [ ]:
# Optional: token-length diagnostics (loads base tokenizer/model)
RUN_TOKEN_STATS = False

if RUN_TOKEN_STATS:
    lens = estimate_qwen_token_percentiles(
        model_name=cfg.model_name,
        max_seq_length=cfg.max_seq_length,
        df=pd.concat([train_df, val_df, test_df], ignore_index=True),
    )
    print(lens)


In [ ]:
# Training resume plan:
# 1) Resume exactly from epoch-2 trainer checkpoint.
# 2) During continuation: save epoch adapters every 3 epochs.
# 3) During continuation: run generation-based eval every 3 epochs on VALIDATION only with BERTScore.
RUN_TRAINING = False
RESUME_FROM_EPOCH2 = True

output_dir = cfg.output_root / "models" / cfg.run_name
resume_checkpoint = output_dir / "checkpoint-764" if RESUME_FROM_EPOCH2 else None

print({"train_rows": len(train_df), "val_rows": len(val_df), "test_rows": len(test_df)})
if resume_checkpoint is not None:
    print("Resume checkpoint:", resume_checkpoint)
    if not resume_checkpoint.exists():
        raise FileNotFoundError(f"Epoch-2 checkpoint not found: {resume_checkpoint}")

if RUN_TRAINING:
    run_info = train_qwen3(
        cfg=cfg,
        ds=ds,
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        include_bertscore_for_epoch_eval=True,
        run_epoch_generation_eval=True,
        run_epoch_test_eval=False,
        checkpoint_every_n_epochs=3,
        generation_eval_every_n_epochs=3,
        resume_from_checkpoint=resume_checkpoint,
        logger=logger,
    )
else:
    print("RUN_TRAINING=False -> evaluation mode only.")
    run_info = {
        "model_name": cfg.model_name,
        "run_name": cfg.run_name,
        "output_dir": str(output_dir),
        "adapter_dir": str(output_dir / "best_adapter"),
        "epoch_adapter_dir": str(output_dir / "epoch_adapters"),
        "eval_dir": str(cfg.output_root / "eval" / cfg.run_name),
    }

run_info



In [ ]:
# Compare BASE (untuned) vs BEST CHECKPOINT SO FAR (tuned)
# - auto-detects best checkpoint from latest trainer_state.json
# - evaluates both on val + test with BERTScore
RUN_COMPARE_BASE_VS_BEST = True

if RUN_COMPARE_BASE_VS_BEST:
    output_dir = cfg.output_root / "models" / cfg.run_name
    ckpts = sorted(
        [p for p in output_dir.glob("checkpoint-*") if p.is_dir()],
        key=lambda p: int(p.name.split("-")[-1]),
    )
    if not ckpts:
        raise FileNotFoundError(f"No checkpoints found in: {output_dir}")

    latest_ckpt = ckpts[-1]
    trainer_state_path = latest_ckpt / "trainer_state.json"
    if not trainer_state_path.exists():
        raise FileNotFoundError(f"Missing trainer_state.json: {trainer_state_path}")

    state = json.loads(trainer_state_path.read_text(encoding="utf-8"))
    best_ckpt_str = state.get("best_model_checkpoint")
    if not best_ckpt_str:
        raise ValueError("best_model_checkpoint is missing in trainer_state.json")

    best_ckpt = Path(best_ckpt_str)
    if not best_ckpt.exists():
        raise FileNotFoundError(f"best_model_checkpoint path does not exist: {best_ckpt}")

    print("Latest checkpoint:", latest_ckpt)
    print("Best checkpoint:", best_ckpt)
    print("Best eval_loss:", state.get("best_metric"))

    prev_wandb = cfg.wandb.enabled
    cfg.wandb.enabled = False  # avoid wandb re-init issues during evaluation

    try:
        base_metrics = evaluate_base_model(
            cfg=cfg,
            val_df=val_df,
            test_df=test_df,
            include_bertscore=True,
            logger=logger,
        )

        best_metrics = evaluate_single_adapter(
            cfg=cfg,
            adapter_dir=best_ckpt,
            tag=f"best_ckpt_{best_ckpt.name}",
            val_df=val_df,
            test_df=test_df,
            include_bertscore=True,
            use_wandb=False,
            logger=logger,
        )
    finally:
        cfg.wandb.enabled = prev_wandb

    comparison_df = pd.DataFrame([base_metrics, best_metrics])
    display(comparison_df)



In [ ]:
# moved



In [ ]:
# Separate generation step (model-agnostic utility)
# This demonstrates using generate_predictions_from_messages independently.
RUN_GENERATION_STEP_ONLY = False

if RUN_GENERATION_STEP_ONLY:
    from experiments.qwen.utils.chat import make_inference_prompt

    adapter_dir = Path(run_info["adapter_dir"])
    model, tokenizer = load_qwen3_model_for_inference(
        model_name=cfg.model_name,
        adapter_dir=adapter_dir if adapter_dir.exists() else None,
        max_seq_length=cfg.max_seq_length,
    )

    demo_pred = generate_predictions_from_messages(
        eval_df=val_df.head(8),
        model=model,
        tokenizer=tokenizer,
        make_inference_prompt_fn=make_inference_prompt,
        parse_score_fn=parse_score,
        parse_rationale_fn=parse_rationale,
        max_seq_length=cfg.max_seq_length,
        max_new_tokens=cfg.generation.max_new_tokens,
        batch_size=4,
    )

    demo_pred.head()
